# Scraper & API Endpoint Verification Notebook

This notebook verifies that the job boards and API endpoints (especially Indian and APAC-focused remote job boards) are working correctly before we deploy to production.

In [ ]:
import sys
import asyncio
from pprint import pprint
import pandas as pd

# Ensure the local 'app' package is importable
if '.' not in sys.path:
    sys.path.append('.')

from app.scraper import (
    scrape_remoteindian,
    scrape_himalayas,
    scrape_weworkremotely,
    scrape_remotive_api,
    scrape_justremote,
    scrape_dailyremote,
    scrape_greenhouse,
    scrape_lever
)
print("Scrapers successfully imported!")

## 1. Test Indian & APAC-Focused Remote Scraper

Let's test `scrape_remoteindian` specifically. This scrapes `remoteindian.com`, which lists remote roles specifically curated for developers in India.

In [ ]:
print("Testing Remote Indian Scraper...")
try:
    # Run the async scraper inside the notebook's event loop
    loop = asyncio.get_event_loop()
    indian_jobs = loop.run_until_complete(scrape_remoteindian(days=10))
    print(f"Success! Found {len(indian_jobs)} remote jobs on Remote Indian.")
    if indian_jobs:
        df_indian = pd.DataFrame([{
            'title': j.title,
            'company': j.company,
            'location': j.location,
            'url': j.url
        } for j in indian_jobs])
        display(df_indian.head(10))
    else:
        print("No jobs found in the specified date range.")
except Exception as e:
    print(f"Error testing Remote Indian scraper: {e}")

## 2. Test Global Remote Scrapers with Strong APAC Listings

Now let's test a batch of global boards (Himalayas, WeWorkRemotely, Remotive, JustRemote, and DailyRemote) that list remote-friendly APAC jobs.

In [ ]:
scrapers = {
    "Himalayas": scrape_himalayas,
    "WeWorkRemotely": scrape_weworkremotely,
    "Remotive": scrape_remotive_api,
    "JustRemote": scrape_justremote,
    "DailyRemote": scrape_dailyremote
}

results = {}
loop = asyncio.get_event_loop()

for name, scraper_fn in scrapers.items():
    print(f"Testing {name}...")
    try:
        jobs = loop.run_until_complete(scraper_fn(days=5))
        results[name] = {"status": "WORKING", "count": len(jobs), "error": "None"}
        print(f"  -> Found {len(jobs)} jobs")
    except Exception as e:
        results[name] = {"status": "FAILED", "count": 0, "error": str(e)}
        print(f"  -> FAILED: {e}")

print("\n--- Global Remote API Test Summary ---")
df_summary = pd.DataFrame.from_dict(results, orient='index')
display(df_summary)